# Baseline inicial — PD‑L1 agonist peptides (Equipo 02)

**Objetivo de este avance:** construir un **modelo de referencia (baseline)** para evaluar la viabilidad del problema de predicción con el dataset inicial.

Este notebook está diseñado para ser **autosuficiente**: carga el dataset, genera features sencillas (composición de aminoácidos + señales numéricas existentes) y evalúa **baselines** con validación cruzada.

**Fecha:** 2026-02-15


## 1) Setup

- Se asume que estás en la carpeta `notebooks/` del repo.
- El dataset base se lee de `../data/raw/dataset.csv` (como en tus avances previos).


In [ ]:
# Librerías base
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import RepeatedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score, precision_recall_curve
)
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import learning_curve, ShuffleSplit

import warnings
warnings.filterwarnings("ignore")

print("OK")


## 2) Carga del dataset

El dataset actual contiene secuencias peptídicas y métricas derivadas del pipeline (ProteinMPNN/FASTA scores y AlphaFold en modo mono y bind).

Ajusta la ruta si tu estructura difiere.


In [ ]:
DATA_PATH = "../data/raw/dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())


## 3) Definir variables objetivo

En tus avances el dataset incluye varias métricas numéricas; típicamente hay dos familias:

- **Mono (plegado del péptido):** `mono_ptm`, `mono_atom_plddt_A_mean`, etc.
- **Bind (interacción con PD‑L1):** `bind_iptm`, `bind_chainpair_iptm_A_B`, etc.

Para un baseline razonable con negocio, suele tener sentido comenzar con una **métrica de binding**.

Aquí dejamos el notebook flexible para que puedas cambiar el objetivo sin reescribir el flujo.


In [ ]:
# Targets sugeridos (ajusta según tu definición de negocio)
candidate_targets = [
    "bind_iptm",
    "bind_chainpair_iptm_A_B",
    "mono_ptm",
    "mono_atom_plddt_A_mean",
]

# Mantén los que sí existan en el CSV
available_targets = [t for t in candidate_targets if t in df.columns]
available_targets


In [ ]:
# === Elige tu variable objetivo aquí ===
TARGET = "bind_iptm" if "bind_iptm" in df.columns else available_targets[0]
print("TARGET =", TARGET)

# Revisa missingness del target
print(df[TARGET].describe())
print("NaNs en target:", df[TARGET].isna().sum())


## 4) Features simples (baseline)

Vamos a usar 2 grupos de features:

**A) Baseline mínimo (solo secuencia):**
- Longitud
- Proporción de cada aminoácido (20 features)

**B) Baseline ampliado (secuencia + señales numéricas disponibles):**
- Todas las numéricas *excepto* el target y columnas de texto

Nota: Si tu objetivo es un score de *binding*, evita incluir features que sean esencialmente el mismo score duplicado (p.ej. `bind_chainpair_iptm_A_B` cuando el target es `bind_iptm`) para reducir fuga de información.


In [ ]:
# --- A) Features de secuencia (composición) ---
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

# Aseguramos que la columna sequence exista
assert "sequence" in df.columns, "No encuentro columna 'sequence' en el CSV."

feat_df = df.copy()
feat_df["length"] = feat_df["sequence"].astype(str).str.len()

for aa in amino_acids:
    feat_df[f"freq_{aa}"] = feat_df["sequence"].astype(str).str.count(aa)
    feat_df[f"prop_{aa}"] = feat_df[f"freq_{aa}"] / feat_df["length"].replace(0, np.nan)

seq_features = ["length"] + [f"prop_{aa}" for aa in amino_acids]

# --- B) Features numéricas existentes (no texto), excluyendo targets y leakage obvio ---
numeric_cols = feat_df.select_dtypes(include=[np.number]).columns.tolist()

# Excluir el target
numeric_cols = [c for c in numeric_cols if c != TARGET]

# Si el target es bind_iptm, suele ser buena idea excluir 'bind_chainpair_iptm_A_B' (muy cercano)
leakage_candidates = []
if TARGET == "bind_iptm" and "bind_chainpair_iptm_A_B" in numeric_cols:
    leakage_candidates.append("bind_chainpair_iptm_A_B")
if TARGET == "bind_chainpair_iptm_A_B" and "bind_iptm" in numeric_cols:
    leakage_candidates.append("bind_iptm")

numeric_cols = [c for c in numeric_cols if c not in leakage_candidates]

# Definir 2 conjuntos de features
X_seq = feat_df[seq_features].copy()
X_all = feat_df[seq_features + numeric_cols].copy()

y = feat_df[TARGET].copy()

print("X_seq:", X_seq.shape, "X_all:", X_all.shape, "y:", y.shape)
print("Leakage excluida:", leakage_candidates)


## 5) Validación cruzada

Con ~40 registros, la varianza del desempeño es alta. Usaremos:

- `RepeatedKFold` (5 folds, 20 repeticiones) para estimar estabilidad.


In [ ]:
cv = RepeatedKFold(n_splits=5, n_repeats=20, random_state=42)


## 6) Baselines de **regresión**

Compararemos:

- **DummyRegressor** (predice el promedio): referencia ingenua
- **Ridge/Lasso**: baseline lineal
- **RandomForestRegressor**: baseline no-lineal

Métricas:

- MAE (principal)
- RMSE
- R²
- Spearman (ranking)


In [ ]:
def spearman_corr(y_true, y_pred):
    # Spearman sin scipy (rank + corr)
    y_true_rank = pd.Series(y_true).rank(method="average").to_numpy()
    y_pred_rank = pd.Series(y_pred).rank(method="average").to_numpy()
    return np.corrcoef(y_true_rank, y_pred_rank)[0, 1]

def regression_report(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    sp = spearman_corr(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Spearman": sp}


In [ ]:
def eval_regressors(X, y, cv, label):
    models = {
        "Dummy(mean)": ("passthrough", DummyRegressor(strategy="mean")),
        "Ridge": (StandardScaler(), Ridge(alpha=1.0, random_state=42)),
        "Lasso": (StandardScaler(), Lasso(alpha=0.001, random_state=42, max_iter=10000)),
        "RF": ("passthrough", RandomForestRegressor(
            n_estimators=400, random_state=42, n_jobs=-1, min_samples_leaf=2
        )),
    }

    rows = []
    for name, (scaler, model) in models.items():
        pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
            ("model", model),
        ])
        y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)
        rep = regression_report(y, y_pred)
        rep.update({"FeatureSet": label, "Model": name})
        rows.append(rep)

    return pd.DataFrame(rows).sort_values("MAE")

reg_seq = eval_regressors(X_seq, y, cv, label="Sequence-only")
reg_all = eval_regressors(X_all, y, cv, label="Sequence + numeric")

display(pd.concat([reg_seq, reg_all], ignore_index=True).sort_values(["FeatureSet", "MAE"]))


### Visual: predicción vs real (mejor modelo por MAE)


In [ ]:
def best_row(dfres):
    return dfres.sort_values("MAE").iloc[0]

best_seq = best_row(reg_seq)
best_all = best_row(reg_all)

best_seq, best_all


In [ ]:
def fit_and_oof_preds(model_name, X, y, cv):
    if model_name == "Dummy(mean)":
        scaler, model = ("passthrough", DummyRegressor(strategy="mean"))
    elif model_name == "Ridge":
        scaler, model = (StandardScaler(), Ridge(alpha=1.0, random_state=42))
    elif model_name == "Lasso":
        scaler, model = (StandardScaler(), Lasso(alpha=0.001, random_state=42, max_iter=10000))
    elif model_name == "RF":
        scaler, model = ("passthrough", RandomForestRegressor(
            n_estimators=400, random_state=42, n_jobs=-1, min_samples_leaf=2
        ))
    else:
        raise ValueError(model_name)

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
        ("model", model),
    ])
    return cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)

y_pred_seq = fit_and_oof_preds(best_seq["Model"], X_seq, y, cv)
y_pred_all = fit_and_oof_preds(best_all["Model"], X_all, y, cv)

plt.figure()
plt.scatter(y, y_pred_seq)
plt.xlabel("Real")
plt.ylabel("Predicción (OOF)")
plt.title(f"Sequence-only | {best_seq['Model']} | MAE={best_seq['MAE']:.4f}")
plt.show()

plt.figure()
plt.scatter(y, y_pred_all)
plt.xlabel("Real")
plt.ylabel("Predicción (OOF)")
plt.title(f"Sequence+numeric | {best_all['Model']} | MAE={best_all['MAE']:.4f}")
plt.show()


## 7) Importancia de características

Usamos **Permutation Importance** (entrenando en todo el dataset solo para interpretabilidad).


In [ ]:
def build_reg_pipe(model_name):
    if model_name == "Dummy(mean)":
        scaler, model = ("passthrough", DummyRegressor(strategy="mean"))
    elif model_name == "Ridge":
        scaler, model = (StandardScaler(), Ridge(alpha=1.0, random_state=42))
    elif model_name == "Lasso":
        scaler, model = (StandardScaler(), Lasso(alpha=0.001, random_state=42, max_iter=10000))
    elif model_name == "RF":
        scaler, model = ("passthrough", RandomForestRegressor(
            n_estimators=600, random_state=42, n_jobs=-1, min_samples_leaf=2
        ))
    else:
        raise ValueError(model_name)

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
        ("model", model),
    ])

chosen = best_all["Model"]
pipe = build_reg_pipe(chosen)
pipe.fit(X_all, y)

perm = permutation_importance(
    pipe, X_all, y,
    n_repeats=200, random_state=42,
    scoring="neg_mean_absolute_error"
)

imp = pd.DataFrame({
    "feature": X_all.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

display(imp.head(20))

plt.figure(figsize=(10,6))
plt.barh(imp.head(20)["feature"][::-1], imp.head(20)["importance_mean"][::-1])
plt.title(f"Permutation importance (top 20) | Model={chosen}")
plt.xlabel("Δ (neg MAE) — mayor = más importante")
plt.show()


In [ ]:
# Coeficientes si el mejor modelo es lineal
if chosen in ["Ridge", "Lasso"]:
    coefs = pipe.named_steps["model"].coef_
    coef_df = pd.DataFrame({"feature": X_all.columns, "coef": coefs})
    coef_df["abs_coef"] = coef_df["coef"].abs()
    display(coef_df.sort_values("abs_coef", ascending=False).head(20))
else:
    print("Modelo no lineal; coeficientes no aplican.")


## 8) ¿Sub/sobreajuste? (learning curve)

Comparación Train vs Validation MAE con `ShuffleSplit` (más estable con pocos datos).


In [ ]:
lc_cv = ShuffleSplit(n_splits=50, test_size=0.2, random_state=42)
pipe = build_reg_pipe(best_all["Model"])

train_sizes, train_scores, valid_scores = learning_curve(
    pipe,
    X_all, y,
    cv=lc_cv,
    train_sizes=np.linspace(0.2, 1.0, 8),
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

train_mae = -train_scores.mean(axis=1)
valid_mae = -valid_scores.mean(axis=1)

plt.figure()
plt.plot(train_sizes, train_mae, marker="o", label="Train MAE")
plt.plot(train_sizes, valid_mae, marker="o", label="Validation MAE")
plt.xlabel("Tamaño de entrenamiento")
plt.ylabel("MAE")
plt.title(f"Learning curve | {best_all['Model']} | TARGET={TARGET}")
plt.legend()
plt.show()


## 9) Baseline como **clasificación** (opcional)

Caso típico de negocio: “¿este candidato es bueno o no?”.
Definimos `good_binder` como **top 25%** del target.


In [ ]:
q = 0.75
thresh = y.quantile(q)
y_bin = (y >= thresh).astype(int)

print(f"Umbral (q={q}): {thresh:.4f} | Positivos: {y_bin.sum()}/{len(y_bin)}")


In [ ]:
def eval_classifiers(X, y_bin, cv, label):
    models = {
        "Dummy(most_frequent)": ("passthrough", DummyClassifier(strategy="most_frequent")),
        "LogReg": (StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)),
        "RF": ("passthrough", RandomForestClassifier(
            n_estimators=500, random_state=42, n_jobs=-1, min_samples_leaf=2, class_weight="balanced"
        )),
    }

    rows = []
    for name, (scaler, model) in models.items():
        pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
            ("model", model),
        ])
        prob = cross_val_predict(pipe, X, y_bin, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
        pred = (prob >= 0.5).astype(int)

        rows.append({
            "FeatureSet": label,
            "Model": name,
            "ROC_AUC": roc_auc_score(y_bin, prob),
            "PR_AUC": average_precision_score(y_bin, prob),
            "F1@0.5": f1_score(y_bin, pred)
        })

    return pd.DataFrame(rows).sort_values("PR_AUC", ascending=False)

cls_seq = eval_classifiers(X_seq, y_bin, cv, "Sequence-only")
cls_all = eval_classifiers(X_all, y_bin, cv, "Sequence + numeric")

display(pd.concat([cls_seq, cls_all], ignore_index=True).sort_values(["FeatureSet", "PR_AUC"], ascending=[True, False]))


### Curva Precision-Recall del mejor clasificador


In [ ]:
best_cls_all = cls_all.sort_values("PR_AUC", ascending=False).iloc[0]
print(best_cls_all)

def build_cls_pipe(model_name):
    if model_name == "Dummy(most_frequent)":
        scaler, model = ("passthrough", DummyClassifier(strategy="most_frequent"))
    elif model_name == "LogReg":
        scaler, model = (StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))
    elif model_name == "RF":
        scaler, model = ("passthrough", RandomForestClassifier(
            n_estimators=700, random_state=42, n_jobs=-1, min_samples_leaf=2, class_weight="balanced"
        ))
    else:
        raise ValueError(model_name)

    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler),
        ("model", model),
    ])

cls_pipe = build_cls_pipe(best_cls_all["Model"])
prob = cross_val_predict(cls_pipe, X_all, y_bin, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
precision, recall, _ = precision_recall_curve(y_bin, prob)

plt.figure()
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall | {best_cls_all['Model']} | PR_AUC={best_cls_all['PR_AUC']:.3f}")
plt.show()


## 10) Métrica adecuada y desempeño mínimo (Precision@K)

Si tu objetivo es priorizar candidatos para validación costosa, suele importar el **ranking**.
Implementamos **Precision@K** usando las predicciones OOF del mejor regresor.


In [ ]:
def precision_at_k(y_true, y_score, k=5, q=0.75):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_score = pd.Series(y_score).reset_index(drop=True)

    pos_thresh = y_true.quantile(q)
    y_pos = (y_true >= pos_thresh).astype(int)

    topk_idx = y_score.sort_values(ascending=False).head(k).index
    return y_pos.loc[topk_idx].mean(), pos_thresh

# Usamos el mejor modelo del set ampliado (regresión)
y_oof = y_pred_all

for k in [3, 5, 10]:
    p_at_k, pos_thresh = precision_at_k(y, y_oof, k=k, q=0.75)
    print(f"Precision@{k} (positivos = top 25% por y_true) = {p_at_k:.3f} | thresh={pos_thresh:.4f}")


## 11) Checklist de respuestas (para tu entrega)

- **¿Qué algoritmo usar como baseline?** Dummy + Ridge/LogReg + RandomForest como no-lineal.
- **¿Importancia de features?** Permutation importance + (coeficientes si es lineal).
- **¿Sub/sobreajuste?** learning curve (train vs val).
- **¿Métrica adecuada?** MAE/RMSE si quieres valor exacto; Spearman/PR‑AUC/Precision@K si quieres priorizar candidatos.
- **¿Desempeño mínimo?** al menos superar Dummy y recuperar candidatos relevantes en top‑K.
